In [ ]:

# ================================================================
# SupplyChainAI — 02 DEMAND FORECASTING
# Single-Cell Jupyter Notebook
# ================================================================
#
# Models:
#   1. Seasonal Naive Baseline
#   2. Random Forest Regressor
#   3. HistGradientBoosting Regressor
#
# Features:
#   - Daily demand aggregation
#   - Missing-date handling
#   - Calendar features
#   - Lag features
#   - Rolling demand statistics
#   - Chronological train / validation / test splits
#   - Model comparison
#   - Recursive 30-day forecasting
#   - Forecast visualization
#   - Feature importance
#   - Model persistence
#   - CSV and JSON report export
#
# Install:
#   pip install pandas numpy matplotlib scikit-learn joblib
#
# IMPORTANT:
#   This notebook forecasts historical units sold.
#   Actual customer demand may exceed observed sales
#   when stockouts or lost sales occur.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ================================================================
# 1. CONFIGURATION
# ================================================================

RANDOM_STATE = 42

FORECAST_HORIZON = 30
VALIDATION_DAYS = 30
TEST_DAYS = 30

LAGS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 14, 28]

MIN_HISTORY_DAYS = 120

DATA_PATH_CANDIDATES = [
    Path("data/processed/sales_clean.csv"),
    Path("../data/processed/sales_clean.csv"),
    Path("data/sales.csv"),
    Path("../data/sales.csv")
]

OUTPUT_ROOT = (
    Path("..")
    if Path("../data").exists() and not Path("data").exists()
    else Path(".")
)

REPORT_DIR = OUTPUT_ROOT / "reports" / "forecasting"
MODEL_DIR = OUTPUT_ROOT / "models" / "demand_forecasting"
OUTPUT_DIR = OUTPUT_ROOT / "data" / "processed"

for directory in [REPORT_DIR, MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 110

print("=" * 72)
print("SUPPLYCHAINAI — DEMAND FORECASTING")
print("=" * 72)

# ================================================================
# 2. LOAD DATA
# ================================================================

DATA_PATH = next(
    (p for p in DATA_PATH_CANDIDATES if p.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Sales data not found. Run 01_data_exploration.ipynb "
        "first, or provide data/sales.csv."
    )

sales = pd.read_csv(DATA_PATH)

required_columns = {
    "date",
    "product_id",
    "quantity_sold"
}

missing_columns = required_columns - set(sales.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

sales["date"] = pd.to_datetime(
    sales["date"],
    errors="coerce"
)

sales["quantity_sold"] = pd.to_numeric(
    sales["quantity_sold"],
    errors="coerce"
)

sales = sales.dropna(
    subset=["date", "quantity_sold", "product_id"]
).copy()

sales = sales[
    sales["quantity_sold"] >= 0
].copy()

if sales.empty:
    raise ValueError("No valid sales observations found.")

sales["date"] = sales["date"].dt.normalize()

print(f"Source: {DATA_PATH}")
print(f"Rows: {len(sales):,}")
print(f"Products: {sales['product_id'].nunique():,}")
print(f"Date range: {sales['date'].min().date()} to "
      f"{sales['date'].max().date()}")

# ================================================================
# 3. DAILY DEMAND AGGREGATION
# ================================================================

daily = (
    sales.groupby("date", as_index=False)
    .agg(demand=("quantity_sold", "sum"))
    .sort_values("date")
)

full_dates = pd.date_range(
    start=daily["date"].min(),
    end=daily["date"].max(),
    freq="D"
)

daily = (
    daily.set_index("date")
    .reindex(full_dates)
    .rename_axis("date")
    .reset_index()
)

missing_days = int(daily["demand"].isna().sum())

print(f"\nMissing calendar days: {missing_days}")

# Assumption:
# Days with no recorded transactions are treated as zero sales.
# This is appropriate only if the source captures all sales days.
daily["demand"] = daily["demand"].fillna(0).astype(float)

if len(daily) < MIN_HISTORY_DAYS:
    raise ValueError(
        f"At least {MIN_HISTORY_DAYS} calendar days are required. "
        f"Found {len(daily)}."
    )

if len(daily) <= (
    max(LAGS)
    + VALIDATION_DAYS
    + TEST_DAYS
):
    raise ValueError("Insufficient history for the selected splits.")

print("\nDaily demand preview:")
print(daily.head().to_string(index=False))

# ================================================================
# 4. FEATURE ENGINEERING
# ================================================================

FEATURE_COLUMNS = [
    "day_of_week",
    "day_of_month",
    "month",
    "quarter",
    "week_of_year",
    "is_weekend",
    "day_of_year_sin",
    "day_of_year_cos",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28"
]

def build_features(df):
    result = df.copy().sort_values("date")

    dates = pd.to_datetime(result["date"])

    result["day_of_week"] = dates.dt.dayofweek
    result["day_of_month"] = dates.dt.day
    result["month"] = dates.dt.month
    result["quarter"] = dates.dt.quarter
    result["week_of_year"] = (
        dates.dt.isocalendar().week.astype(int)
    )
    result["is_weekend"] = (
        dates.dt.dayofweek >= 5
    ).astype(int)

    day_of_year = dates.dt.dayofyear

    result["day_of_year_sin"] = np.sin(
        2 * np.pi * day_of_year / 365.25
    )

    result["day_of_year_cos"] = np.cos(
        2 * np.pi * day_of_year / 365.25
    )

    shifted = result["demand"].shift(1)

    for lag in LAGS:
        result[f"lag_{lag}"] = (
            result["demand"].shift(lag)
        )

    for window in ROLLING_WINDOWS:
        result[f"rolling_mean_{window}"] = (
            shifted.rolling(window).mean()
        )

        result[f"rolling_std_{window}"] = (
            shifted.rolling(window).std()
        )

    return result


feature_data = build_features(daily)

feature_data = feature_data.dropna(
    subset=FEATURE_COLUMNS
).reset_index(drop=True)

print(f"\nFeature dataset rows: {len(feature_data)}")
print(f"Feature count: {len(FEATURE_COLUMNS)}")

# ================================================================
# 5. CHRONOLOGICAL SPLIT
# ================================================================

test_start = len(daily) - TEST_DAYS
validation_start = test_start - VALIDATION_DAYS

validation_start_date = daily.iloc[
    validation_start
]["date"]

test_start_date = daily.iloc[
    test_start
]["date"]

train_df = feature_data[
    feature_data["date"] < validation_start_date
].copy()

validation_df = feature_data[
    (feature_data["date"] >= validation_start_date) &
    (feature_data["date"] < test_start_date)
].copy()

test_df = feature_data[
    feature_data["date"] >= test_start_date
].copy()

if min(
    len(train_df),
    len(validation_df),
    len(test_df)
) == 0:
    raise ValueError("One or more chronological splits are empty.")

print("\nChronological splits:")
print(f"Train:      {len(train_df)} days")
print(f"Validation: {len(validation_df)} days")
print(f"Test:       {len(test_df)} days")

# ================================================================
# 6. METRICS
# ================================================================

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    predicted = np.maximum(predicted, 0)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    denominator = np.abs(actual).sum()

    wape = (
        np.abs(actual - predicted).sum()
        / denominator * 100
        if denominator > 0
        else np.nan
    )

    bias = np.mean(predicted - actual)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "WAPE_pct": float(wape),
        "Bias": float(bias)
    }


def seasonal_naive_predictions(history, horizon):
    """
    Recursive weekly seasonal-naive forecast.
    Uses the last 7 known values and repeats them.
    """
    values = list(np.asarray(history, dtype=float))
    predictions = []

    for _ in range(horizon):
        prediction = max(0.0, values[-7])
        predictions.append(prediction)
        values.append(prediction)

    return np.asarray(predictions)


# ================================================================
# 7. RECURSIVE FORECAST ENGINE
# ================================================================

def recursive_forecast(model, history_df, horizon):
    """
    Generate future daily predictions without using
    actual future target values.

    history_df must contain columns date and demand.
    """

    working = history_df[
        ["date", "demand"]
    ].copy().sort_values("date").reset_index(drop=True)

    predictions = []

    for _ in range(horizon):
        next_date = (
            working["date"].iloc[-1]
            + pd.Timedelta(days=1)
        )

        future_row = pd.DataFrame([{
            "date": next_date,
            "demand": np.nan
        }])

        temp = pd.concat(
            [working, future_row],
            ignore_index=True
        )

        features = build_features(temp)

        X_next = features.iloc[[-1]][FEATURE_COLUMNS]

        if X_next.isna().any().any():
            raise ValueError(
                "Insufficient historical demand for forecasting."
            )

        prediction = float(
            model.predict(X_next)[0]
        )

        prediction = max(0.0, prediction)

        predictions.append({
            "date": next_date,
            "predicted_demand": prediction
        })

        working = pd.concat([
            working,
            pd.DataFrame([{
                "date": next_date,
                "demand": prediction
            }])
        ], ignore_index=True)

    return pd.DataFrame(predictions)


# ================================================================
# 8. TRAIN CANDIDATE MODELS
# ================================================================

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    )
}

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["demand"]

validation_history = daily[
    daily["date"] < validation_start_date
][["date", "demand"]]

validation_actual = validation_df[
    ["date", "demand"]
].reset_index(drop=True)

results = []
trained_models = {}
validation_forecasts = {}

# Seasonal-naive baseline
naive_preds = seasonal_naive_predictions(
    validation_history["demand"],
    len(validation_actual)
)

baseline_metrics = calculate_metrics(
    validation_actual["demand"],
    naive_preds
)

results.append({
    "model": "SeasonalNaive",
    **baseline_metrics
})

validation_forecasts["SeasonalNaive"] = naive_preds

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    forecast_df = recursive_forecast(
        model,
        validation_history,
        len(validation_actual)
    )

    predictions = forecast_df[
        "predicted_demand"
    ].to_numpy()

    metrics = calculate_metrics(
        validation_actual["demand"],
        predictions
    )

    results.append({
        "model": name,
        **metrics
    })

    trained_models[name] = model
    validation_forecasts[name] = predictions

results_df = pd.DataFrame(results).sort_values(
    ["MAE", "RMSE"]
).reset_index(drop=True)

print("\nVALIDATION MODEL COMPARISON")
print(results_df.round(3).to_string(index=False))

best_model_name = results_df.iloc[0]["model"]

print(f"\nSelected model: {best_model_name}")

# ================================================================
# 9. VALIDATION VISUALIZATION
# ================================================================

plt.figure(figsize=(14, 6))

plt.plot(
    validation_actual["date"],
    validation_actual["demand"],
    label="Actual",
    linewidth=2
)

for name, predictions in validation_forecasts.items():
    plt.plot(
        validation_actual["date"],
        predictions,
        label=name,
        alpha=0.85
    )

plt.title("Demand Forecasting — Validation Period")
plt.xlabel("Date")
plt.ylabel("Daily Units Sold")
plt.legend()
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "validation_comparison.png"
)
plt.show()

# ================================================================
# 10. FINAL HOLDOUT TEST
# ================================================================

# Refit only the selected ML model using data before
# the test period. The test targets remain unseen.

pretest_features = feature_data[
    feature_data["date"] < test_start_date
]

test_history = daily[
    daily["date"] < test_start_date
][["date", "demand"]]

if best_model_name == "SeasonalNaive":
    test_predictions = seasonal_naive_predictions(
        test_history["demand"],
        len(test_df)
    )
    final_model = None

else:
    if best_model_name == "RandomForest":
        final_model = RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    else:
        final_model = HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=RANDOM_STATE
        )

    final_model.fit(
        pretest_features[FEATURE_COLUMNS],
        pretest_features["demand"]
    )

    test_predictions = recursive_forecast(
        final_model,
        test_history,
        len(test_df)
    )["predicted_demand"].to_numpy()

test_metrics = calculate_metrics(
    test_df["demand"],
    test_predictions
)

print("\nFINAL HOLDOUT TEST METRICS")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.3f}")

test_results = test_df[
    ["date", "demand"]
].copy()

test_results["predicted_demand"] = test_predictions
test_results["absolute_error"] = np.abs(
    test_results["demand"] -
    test_results["predicted_demand"]
)

test_results.to_csv(
    REPORT_DIR / "test_predictions.csv",
    index=False
)

plt.figure(figsize=(14, 5))

plt.plot(
    test_results["date"],
    test_results["demand"],
    label="Actual",
    linewidth=2
)

plt.plot(
    test_results["date"],
    test_results["predicted_demand"],
    label=f"Forecast ({best_model_name})",
    linewidth=2
)

plt.title("Final Holdout Test — Actual vs Forecast")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.tight_layout()

plt.savefig(REPORT_DIR / "test_forecast.png")
plt.show()

# ================================================================
# 11. TRAIN FINAL PRODUCTION-CANDIDATE MODEL
# ================================================================

# Retrain using all available observations after evaluation.
# This model is intended for subsequent integration/testing.

if best_model_name == "SeasonalNaive":
    production_model = None

else:
    if best_model_name == "RandomForest":
        production_model = RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    else:
        production_model = HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=RANDOM_STATE
        )

    production_model.fit(
        feature_data[FEATURE_COLUMNS],
        feature_data["demand"]
    )

# ================================================================
# 12. FORECAST NEXT 30 DAYS
# ================================================================

if production_model is None:
    future_predictions = seasonal_naive_predictions(
        daily["demand"],
        FORECAST_HORIZON
    )

    future_dates = pd.date_range(
        daily["date"].max() + pd.Timedelta(days=1),
        periods=FORECAST_HORIZON
    )

    future_forecast = pd.DataFrame({
        "date": future_dates,
        "predicted_demand": future_predictions
    })

else:
    future_forecast = recursive_forecast(
        production_model,
        daily,
        FORECAST_HORIZON
    )

future_forecast["predicted_demand"] = (
    future_forecast["predicted_demand"]
    .clip(lower=0)
    .round(2)
)

# These are point forecasts, not prediction intervals.
# Do not interpret them as guaranteed demand levels.

print("\nNEXT 30-DAY FORECAST")
print(future_forecast.to_string(index=False))

# ================================================================
# 13. FUTURE FORECAST VISUALIZATION
# ================================================================

history_window = daily.tail(60)

plt.figure(figsize=(14, 6))

plt.plot(
    history_window["date"],
    history_window["demand"],
    label="Historical Demand",
    linewidth=2
)

plt.plot(
    future_forecast["date"],
    future_forecast["predicted_demand"],
    label="30-Day Forecast",
    linewidth=2,
    linestyle="--"
)

plt.axvline(
    daily["date"].max(),
    linestyle=":",
    color="gray",
    label="Forecast Start"
)

plt.title(
    f"SupplyChainAI — 30-Day Demand Forecast ({best_model_name})"
)

plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "future_30_day_forecast.png"
)
plt.show()

# ================================================================
# 14. FEATURE IMPORTANCE
# ================================================================

if (
    production_model is not None
    and hasattr(production_model, "feature_importances_")
):
    feature_importance = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance": production_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nFEATURE IMPORTANCE")
    print(feature_importance.to_string(index=False))

    plt.figure(figsize=(11, 7))

    plt.barh(
        feature_importance["feature"].iloc[::-1],
        feature_importance["importance"].iloc[::-1]
    )

    plt.title("Random Forest Feature Importance")
    plt.xlabel("Importance")
    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "feature_importance.png"
    )
    plt.show()

else:
    print(
        "\nNative feature importance is unavailable "
        "for the selected model."
    )

# ================================================================
# 15. BUSINESS FORECAST KPIs
# ================================================================

historical_daily_average = float(
    daily["demand"].tail(30).mean()
)

forecast_daily_average = float(
    future_forecast["predicted_demand"].mean()
)

forecast_total = float(
    future_forecast["predicted_demand"].sum()
)

forecast_change_pct = (
    (
        forecast_daily_average
        - historical_daily_average
    )
    / historical_daily_average
    * 100
    if historical_daily_average > 0
    else np.nan
)

forecast_kpis = pd.DataFrame([
    {
        "metric": "Forecast Horizon",
        "value": FORECAST_HORIZON
    },
    {
        "metric": "Historical Daily Average (30d)",
        "value": round(historical_daily_average, 2)
    },
    {
        "metric": "Forecast Daily Average",
        "value": round(forecast_daily_average, 2)
    },
    {
        "metric": "Forecast Total Units",
        "value": round(forecast_total, 2)
    },
    {
        "metric": "Forecast Change %",
        "value": round(forecast_change_pct, 2)
    }
])

print("\nFORECAST BUSINESS KPIs")
print(forecast_kpis.to_string(index=False))

# ================================================================
# 16. EXPORT MODEL
# ================================================================

model_artifact = {
    "project": "SupplyChainAI",
    "model_name": best_model_name,
    "model": production_model,
    "feature_columns": FEATURE_COLUMNS,
    "lags": LAGS,
    "rolling_windows": ROLLING_WINDOWS,
    "forecast_horizon": FORECAST_HORIZON,
    "training_end_date": str(
        daily["date"].max().date()
    ),
    "historical_demand": daily[
        ["date", "demand"]
    ].tail(60).copy(),
    "test_metrics": test_metrics,
    "random_state": RANDOM_STATE
}

MODEL_PATH = (
    MODEL_DIR / "demand_forecasting_model.joblib"
)

joblib.dump(model_artifact, MODEL_PATH)

# Load joblib files only from trusted sources.
# Deserialization can execute arbitrary code.

# ================================================================
# 17. EXPORT FORECAST AND REPORT
# ================================================================

future_forecast.to_csv(
    OUTPUT_DIR / "demand_forecast_30d.csv",
    index=False
)

results_df.to_csv(
    REPORT_DIR / "model_comparison.csv",
    index=False
)

forecast_kpis.to_csv(
    REPORT_DIR / "forecast_kpis.csv",
    index=False
)

def json_number(value):
    value = float(value)
    return value if np.isfinite(value) else None

report = {
    "project": "SupplyChainAI",
    "notebook": "02_demand_forecasting.ipynb",
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "data_source": str(DATA_PATH),
    "model": best_model_name,
    "forecast_horizon_days": FORECAST_HORIZON,
    "training_end_date": str(
        daily["date"].max().date()
    ),
    "validation_metrics": {
        row["model"]: {
            metric: json_number(row[metric])
            for metric in [
                "MAE", "RMSE", "WAPE_pct", "Bias"
            ]
        }
        for _, row in results_df.iterrows()
    },
    "test_metrics": {
        metric: json_number(value)
        for metric, value in test_metrics.items()
    },
    "forecast": {
        "total_units": json_number(forecast_total),
        "average_daily_units": json_number(
            forecast_daily_average
        ),
        "change_pct": json_number(
            forecast_change_pct
        )
    },
    "limitations": [
        "Observed sales are used as a proxy for demand.",
        "Missing calendar days are assumed to have zero sales.",
        "Forecasts are aggregate across products and warehouses.",
        "Only calendar and historical demand features are used.",
        "Point forecasts do not include uncertainty intervals.",
        "Model selection uses a single chronological validation window."
    ]
}

with open(
    REPORT_DIR / "forecast_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=2)

# ================================================================
# 18. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 72)
print("SUPPLYCHAINAI — FORECASTING COMPLETED")
print("=" * 72)

print(f"Selected model: {best_model_name}")
print(f"Test MAE: {test_metrics['MAE']:.2f}")
print(f"Test RMSE: {test_metrics['RMSE']:.2f}")
print(f"Test WAPE: {test_metrics['WAPE_pct']:.2f}%")
print(f"Forecast horizon: {FORECAST_HORIZON} days")
print(f"Forecast units: {forecast_total:,.0f}")

print("\nSaved files:")
print(f"Model:   {MODEL_PATH.resolve()}")
print(
    "Forecast:",
    (OUTPUT_DIR / "demand_forecast_30d.csv").resolve()
)
print(f"Reports: {REPORT_DIR.resolve()}")

print("\nNext notebooks:")
print("03_inventory_optimization.ipynb")
print("04_supplier_risk_model.ipynb")
print("05_logistics_optimization.ipynb")

print("=" * 72)
